# 01 - Exploratory data analysis

**Goal:** understand the German Credit data well enough to justify every modelling decision that follows.
Run from the repository root's environment (`pip install -e .`); open in VS Code with the Jupyter extension.

**Contents:** overview → missing values → numeric features → categorical features → sensitive attributes → key insights.

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from creditlens.config import NO_ACCOUNT, RAW_DATA_PATH
from creditlens.data import load_clean

sns.set_theme(style="whitegrid", context="notebook")
df = load_clean(RAW_DATA_PATH)
df["bad"] = (df["risk"] == "bad").astype(int)  # 1 = bad credit risk (the event we predict)
df["checking"] = df["checking_account"].fillna(NO_ACCOUNT)
df["saving"] = df["saving_accounts"].fillna(NO_ACCOUNT)
print(df.shape)
df.head()

## 1. Overview and target balance

In [ ]:
display(df.dtypes.to_frame("dtype").T)
display(df.describe().round(1))
print(f"Bad-risk rate: {df.bad.mean():.1%}  ({df.bad.sum()} of {len(df)})")

The target is moderately imbalanced (70/30). Accuracy alone would be misleading - a model predicting *good* for everyone scores 70%. That motivates ROC-AUC/PR-AUC plus a **cost-based** decision rule.

## 2. Missing values
Only the two account columns have missing values. Are they missing at random, or informative?

In [ ]:
miss = df[["saving_accounts", "checking_account"]].isna().mean().mul(100).round(1)
print("% missing:\n", miss.to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, col, name in zip(
    ax, ["checking", "saving"], ["Checking account", "Savings account"], strict=False
):
    order = [NO_ACCOUNT, "little", "moderate", "quite rich", "rich"]
    order = [o for o in order if o in df[col].unique()]
    rate = df.groupby(col).bad.mean().reindex(order)
    n = df[col].value_counts().reindex(order)
    sns.barplot(x=rate.index, y=rate.values, ax=a, color="#1f5fa8")
    for i, (r, k) in enumerate(zip(rate.values, n.values, strict=False)):
        a.text(i, r + 0.01, f"{r:.0%}\nn={k}", ha="center", fontsize=8)
    a.axhline(df.bad.mean(), ls="--", color="grey")
    a.set(title=f"Bad-risk rate by {name.lower()}", ylabel="P(bad)", xlabel="", ylim=(0, 0.6))
plt.tight_layout()
plt.show()

**Missing is not random.** Applicants with *no checking account* (39% of the data) are the **safest** group (~12% bad), far below those with a 'little' balance (~49%). Imputing the mode would destroy this signal, so `NaN` becomes an explicit `no_account` category.

## 3. Numeric features

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
sns.histplot(df, x="credit_amount", hue="risk", bins=40, ax=ax[0, 0], element="step")
ax[0, 0].set_title(f"Credit amount (skew {df.credit_amount.skew():.2f})")
sns.histplot(x=np.log1p(df.credit_amount), hue=df.risk, bins=40, ax=ax[0, 1], element="step")
ax[0, 1].set_title(f"log(1 + credit amount) (skew {np.log1p(df.credit_amount).skew():.2f})")
sns.histplot(df, x="duration", hue="risk", bins=25, ax=ax[0, 2], element="step")
ax[0, 2].set_title("Duration (months)")

dur = df.groupby(
    pd.cut(df.duration, [0, 12, 24, 36, 100], labels=["<=12", "13-24", "25-36", ">36"])
).bad.agg(["mean", "size"])
sns.barplot(x=dur.index, y=dur["mean"], ax=ax[1, 0], color="#c0392b")
ax[1, 0].set(title="Bad rate by duration", ylabel="P(bad)", xlabel="months")
amt = df.groupby(pd.qcut(df.credit_amount, 5)).bad.mean()
sns.barplot(x=[f"Q{i + 1}" for i in range(5)], y=amt.values, ax=ax[1, 1], color="#c0392b")
ax[1, 1].set(title="Bad rate by credit-amount quintile", ylabel="P(bad)")
sns.scatterplot(data=df, x="duration", y="credit_amount", hue="risk", alpha=0.5, ax=ax[1, 2])
ax[1, 2].set(title=f"Amount vs duration (corr {df.credit_amount.corr(df.duration):.2f})")
plt.tight_layout()
plt.show()

print(
    df[["age", "job", "credit_amount", "duration", "bad"]]
    .corr()["bad"]
    .round(3)
    .drop("bad")
    .to_string()
)

* `credit_amount` is strongly right-skewed (1.95) → **log transform** (skew 0.13).
* Risk rises **monotonically with duration**: ~21% bad up to 12 months → ~52% beyond 36 months.
* Amount matters mainly in the top quintile (~43%); amount and duration are correlated (0.62), so the repayment burden `amount / duration` is a natural engineered feature.

## 4. Categorical features

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for a, col, title in zip(
    ax.ravel(),
    ["purpose", "housing", "job", "saving"],
    ["Purpose", "Housing", "Job level (0=unskilled ... 3=highly skilled)", "Savings account"],
    strict=False,
):
    rate = df.groupby(col).bad.agg(["mean", "size"]).sort_values("mean")
    sns.barplot(x=rate["mean"], y=rate.index.astype(str), ax=a, color="#1f5fa8")
    for i, (r, k) in enumerate(zip(rate["mean"], rate["size"], strict=False)):
        a.text(r + 0.005, i, f"{r:.0%} (n={k})", va="center", fontsize=8)
    a.axvline(df.bad.mean(), ls="--", color="grey")
    a.set(title=title, xlabel="P(bad)", ylabel="", xlim=(0, 0.6))
plt.tight_layout()
plt.show()

* **Housing:** owners ~26% bad vs renters ~39% and 'free' housing ~41%.
* **Purpose:** radio/TV is safest (~22%); vacation/others and education look riskiest but have tiny samples (12 and 59 rows) - beware overinterpreting.
* **Job level** barely separates the classes (28-35%) → kept as a single ordinal feature rather than one-hot encoded.
* **Savings:** clear gradient - 'rich' ~12% vs 'little' ~36%.

## 5. Sensitive attributes (context for the fairness audit)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
sex = df.groupby("sex").bad.mean()
sns.barplot(x=sex.index, y=sex.values, ax=ax[0], color="#7f8c8d")
ax[0].set(title="Bad rate by sex", ylabel="P(bad)", ylim=(0, 0.5))
age = df.groupby(
    pd.cut(df.age, [0, 25, 35, 50, 100], labels=["<=25", "26-35", "36-50", ">50"])
).bad.mean()
sns.barplot(x=age.index, y=age.values, ax=ax[1], color="#7f8c8d")
ax[1].set(title="Bad rate by age group", ylabel="P(bad)", ylim=(0, 0.5))
plt.tight_layout()
plt.show()
print(df.groupby("sex").size().to_dict())

Base rates differ by sex (F 35% vs M 28%) and strongly by age (≤25: 42% vs 36-50: 24%). These are **historical outcomes in a small sample**, not causal facts - and using such attributes for lending decisions is legally and ethically fraught. See `reports/fairness_audit.md`.

## 6. Key insights → modelling decisions

| # | Insight | Decision |
|---|---|---|
| 1 | 70/30 imbalance; errors have unequal business cost | Use ROC/PR-AUC + a **cost matrix** (FN=5, FP=1) and pick the threshold by expected cost |
| 2 | Missing account info is *informative* (no checking account → safest group) | Encode as explicit `no_account` category, never impute |
| 3 | Duration has a monotonic, strong effect | Keep numeric; tree models can capture non-linearity |
| 4 | Amount is heavily skewed and correlated with duration | `log_credit_amount` + `monthly_payment` features |
| 5 | Job level is weak | Single ordinal feature |
| 6 | Several small categories (n=12, 22) | Fixed category lists; unseen values handled at serving; regularised models |
| 7 | Age and sex relate to outcome but are protected | Evaluate with a fairness audit; decide on inclusion from the results |
| 8 | Only 1,000 rows | Repeated stratified CV, hold-out set touched once, bootstrap CIs |